# 🌾 Agriculture Data Warehouse: Raw → Staging → Star Schema, with Data Tests

This notebook reproduces the transformation layer of my **Agriculture Data Warehouse** project so it runs end to end on Kaggle.

**The original project**

| Layer | Tool | What it does |
|---|---|---|
| Ingestion | Apache NiFi | `GetFile` → `PutDatabaseRecord` moves incoming CSVs into `raw.farm_yield` |
| Warehouse | SQL Server | `raw` → `stg` → `mart` schemas |
| Transformation | dbt | Staging view + star schema (5 dimensions, 1 fact) with schema tests |
| Orchestration | Apache Airflow (Docker) | Triggers NiFi and runs `dbt run` / `dbt test` |
| Reporting | Power BI | Dashboard on the `mart` schema |

**In this notebook**, SQLite stands in for SQL Server and plain Python stands in for NiFi and Airflow. The SQL is the same as the dbt models, with small dialect changes, and the dbt tests from `schema.yml` are turned into SQL checks.

> ⚠️ **About the data:** the project runs on **synthetic** farm-yield data made by a mock-data generator (seed 42), not real agricultural measurements. The goal here is the pipeline and data-quality engineering, not agronomy insights. Section 6 shows why that matters.

## 1. Setup

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
con = sqlite3.connect(":memory:")
print("SQLite version:", sqlite3.sqlite_version)

## 2. Source data: mock generator

This is the same generator the project uses to drop `farm_yield_data.csv` into NiFi's watched folder.

In [ ]:
np.random.seed(42)
num_records = 1000

crops = ['Wheat', 'Rice', 'Corn', 'Soybeans', 'Cotton', 'Sugarcane']
regions = ['North', 'South', 'East', 'West', 'Central']
soils = ['Clay', 'Sandy', 'Loamy', 'Silt', 'Peaty']
irrigations = ['Drip', 'Sprinkler', 'Flood', 'Manual']
seasons = ['Kharif', 'Rabi', 'Summer', 'Monsoon']

source_df = pd.DataFrame({
    'Farm_ID': np.random.randint(1001, 2000, size=num_records),
    'Crop_Type': np.random.choice(crops, size=num_records),
    'Region': np.random.choice(regions, size=num_records),
    'Soil_Type': np.random.choice(soils, size=num_records),
    'Irrigation_Type': np.random.choice(irrigations, size=num_records),
    'Fertilizer_Used': np.random.uniform(50, 500, size=num_records).round(2),
    'Pesticide_Used': np.random.uniform(5, 50, size=num_records).round(2),
    'Water_Usage': np.random.uniform(100, 1000, size=num_records).round(2),
    'Rainfall': np.random.uniform(200, 1500, size=num_records).round(2),
    'Temperature': np.random.uniform(15, 45, size=num_records).round(2),
    'Harvest_Season': np.random.choice(seasons, size=num_records),
    'Yield': np.random.uniform(1, 15, size=num_records).round(2),
})
source_df.head()

The generator's output is perfectly clean, so the staging layer would have nothing to fix. To exercise the cleaning logic, I append a few **deliberately dirty rows** of the kind real CSV feeds contain: stray whitespace, inconsistent casing, and a missing `Farm_ID`.

In [ ]:
dirty_rows = pd.DataFrame([
    {'Farm_ID': 1500,   'Crop_Type': '  wheat ', 'Region': 'north',    'Soil_Type': 'LOAMY ', 'Irrigation_Type': ' drip',
     'Fertilizer_Used': 210.5, 'Pesticide_Used': 12.3, 'Water_Usage': 450.0, 'Rainfall': 800.0, 'Temperature': 24.1,
     'Harvest_Season': 'rabi ', 'Yield': 7.8},
    {'Farm_ID': 1501,   'Crop_Type': 'Rice',     'Region': ' South ',  'Soil_Type': 'clay',   'Irrigation_Type': 'FLOOD',
     'Fertilizer_Used': 330.0, 'Pesticide_Used': 20.0, 'Water_Usage': 900.0, 'Rainfall': 1200.0, 'Temperature': 31.0,
     'Harvest_Season': 'Kharif', 'Yield': 9.1},
    {'Farm_ID': None,   'Crop_Type': 'Corn',     'Region': 'East',     'Soil_Type': 'Silt',   'Irrigation_Type': 'Manual',
     'Fertilizer_Used': 120.0, 'Pesticide_Used': 8.0,  'Water_Usage': 300.0, 'Rainfall': 600.0, 'Temperature': 28.0,
     'Harvest_Season': 'Summer', 'Yield': 5.5},
])
raw_input_df = pd.concat([source_df, dirty_rows], ignore_index=True)
print(f"{len(source_df)} generated rows + {len(dirty_rows)} dirty rows = {len(raw_input_df)} rows")
print("Distinct Region spellings in raw input:", sorted(raw_input_df['Region'].unique(), key=str))

## 3. Raw layer (the NiFi step)

NiFi's `PutDatabaseRecord` writes each CSV record as-is into `raw.farm_yield`, adding a load timestamp. There are no transformations at this layer.

In [ ]:
raw = raw_input_df.copy()
raw['_loaded_at'] = pd.Timestamp('2026-01-01 08:00:00').isoformat()
raw.to_sql('raw_farm_yield', con, index=False)

pd.read_sql("SELECT COUNT(*) AS raw_rows FROM raw_farm_yield", con)

## 4. Staging layer: `stg_farm_yield`

This translates `models/staging/stg_farm_yield.sql`. It standardizes text (`UPPER(TRIM())`), casts types, renames columns with units, and drops rows that have no `Farm_ID`.

*Dialect note: SQL Server `DECIMAL(10,2)` becomes `ROUND(CAST(... AS REAL), 2)` in SQLite.*

In [ ]:
con.executescript('''
CREATE VIEW stg_farm_yield AS
WITH source AS (
    SELECT * FROM raw_farm_yield
),
renamed AS (
    SELECT
        CAST(Farm_ID AS INT)                        AS farm_id,
        UPPER(TRIM(Crop_Type))                      AS crop_type,
        UPPER(TRIM(Region))                         AS region_name,
        UPPER(TRIM(Soil_Type))                      AS soil_type,
        UPPER(TRIM(Irrigation_Type))                AS irrigation_type,
        ROUND(CAST(Fertilizer_Used AS REAL), 2)     AS fertilizer_used_kg,
        ROUND(CAST(Pesticide_Used AS REAL), 2)      AS pesticide_used_kg,
        ROUND(CAST(Water_Usage AS REAL), 2)         AS water_usage_m3,
        ROUND(CAST(Rainfall AS REAL), 2)            AS rainfall_mm,
        ROUND(CAST(Temperature AS REAL), 2)         AS temperature_c,
        UPPER(TRIM(Harvest_Season))                 AS harvest_season,
        ROUND(CAST(Yield AS REAL), 2)               AS yield_tons,
        _loaded_at                                  AS raw_loaded_at
    FROM source
    WHERE Farm_ID IS NOT NULL
)
SELECT * FROM renamed;
''')

print("Rows raw → staging:",
      pd.read_sql("SELECT COUNT(*) n FROM raw_farm_yield", con).n[0], "→",
      pd.read_sql("SELECT COUNT(*) n FROM stg_farm_yield", con).n[0])
print("Distinct regions after cleaning:",
      pd.read_sql("SELECT DISTINCT region_name FROM stg_farm_yield ORDER BY 1", con).region_name.tolist())

The three dirty rows are handled: whitespace and casing are normalized, so `' South '` and `'north'` merge into the existing `SOUTH` and `NORTH`, and the row without a `Farm_ID` is dropped.

## 5. Mart layer: star schema

These translate the five dimension models and the fact model from `models/marts/`. Each dimension gets a surrogate key from `ROW_NUMBER()`. The fact table joins the staging rows to every dimension.

```
                 dim_farm
                    │
 dim_crop ── fact_farm_yield ── dim_region
 (crop × season)    │    │
               dim_soil  dim_irrigation
```

In [ ]:
dims = {
    'dim_farm': '''
        WITH staging AS (SELECT DISTINCT farm_id FROM stg_farm_yield)
        SELECT ROW_NUMBER() OVER (ORDER BY farm_id) AS farm_key, farm_id
        FROM staging''',
    'dim_crop': '''
        WITH staging AS (SELECT DISTINCT crop_type, harvest_season FROM stg_farm_yield)
        SELECT ROW_NUMBER() OVER (ORDER BY crop_type, harvest_season) AS crop_key,
               crop_type, harvest_season
        FROM staging''',
    'dim_region': '''
        WITH staging AS (SELECT DISTINCT region_name FROM stg_farm_yield)
        SELECT ROW_NUMBER() OVER (ORDER BY region_name) AS region_key, region_name
        FROM staging''',
    'dim_soil': '''
        WITH staging AS (SELECT DISTINCT soil_type FROM stg_farm_yield)
        SELECT ROW_NUMBER() OVER (ORDER BY soil_type) AS soil_key, soil_type
        FROM staging''',
    'dim_irrigation': '''
        WITH staging AS (SELECT DISTINCT irrigation_type FROM stg_farm_yield)
        SELECT ROW_NUMBER() OVER (ORDER BY irrigation_type) AS irrigation_key, irrigation_type
        FROM staging''',
}
for name, sql in dims.items():
    con.execute(f"CREATE TABLE {name} AS {sql}")

con.execute('''
CREATE TABLE fact_farm_yield AS
SELECT
    f.farm_key, c.crop_key, r.region_key, s.soil_key, i.irrigation_key,
    stg.fertilizer_used_kg, stg.pesticide_used_kg, stg.water_usage_m3,
    stg.rainfall_mm, stg.temperature_c, stg.yield_tons
FROM stg_farm_yield stg
JOIN dim_farm       f ON stg.farm_id = f.farm_id
JOIN dim_crop       c ON stg.crop_type = c.crop_type AND stg.harvest_season = c.harvest_season
JOIN dim_region     r ON stg.region_name = r.region_name
JOIN dim_soil       s ON stg.soil_type = s.soil_type
JOIN dim_irrigation i ON stg.irrigation_type = i.irrigation_type
''')

layers = ['raw_farm_yield', 'stg_farm_yield', *dims, 'fact_farm_yield']
pd.DataFrame({
    'model': layers,
    'rows': [pd.read_sql(f"SELECT COUNT(*) n FROM {t}", con).n[0] for t in layers],
})

## 6. Data tests (the `dbt test` step)

The same tests declared in `schema.yml` (`not_null`, `unique` and `relationships`) are compiled to SQL. As in dbt, each test query returns the **failing rows**, so a test passes when it returns zero rows.

I also add two checks that the dbt project didn't have:
- **Row reconciliation:** the fact table must keep every staging row. An inner join silently drops rows whose dimension lookup fails.
- **Accepted range:** yield must be positive.

In [ ]:
schema_tests = {
    'stg_farm_yield': {'farm_id': ['not_null'], 'yield_tons': ['not_null']},
    'dim_farm':       {'farm_key': ['unique', 'not_null'], 'farm_id': ['unique', 'not_null']},
    'dim_crop':       {'crop_key': ['unique', 'not_null'], 'crop_type': ['not_null'], 'harvest_season': ['not_null']},
    'dim_region':     {'region_key': ['unique', 'not_null'], 'region_name': ['unique', 'not_null']},
    'dim_soil':       {'soil_key': ['unique', 'not_null'], 'soil_type': ['unique', 'not_null']},
    'dim_irrigation': {'irrigation_key': ['unique', 'not_null'], 'irrigation_type': ['unique', 'not_null']},
    'fact_farm_yield': {
        'farm_key':       ['not_null', ('relationships', 'dim_farm')],
        'crop_key':       ['not_null', ('relationships', 'dim_crop')],
        'region_key':     ['not_null', ('relationships', 'dim_region')],
        'soil_key':       ['not_null', ('relationships', 'dim_soil')],
        'irrigation_key': ['not_null', ('relationships', 'dim_irrigation')],
        'yield_tons':     ['not_null'],
    },
}

def compile_test(model, column, test):
    if test == 'not_null':
        return f"SELECT * FROM {model} WHERE {column} IS NULL"
    if test == 'unique':
        return f"SELECT {column} FROM {model} GROUP BY {column} HAVING COUNT(*) > 1"
    kind, parent = test
    assert kind == 'relationships'
    return (f"SELECT c.{column} FROM {model} c LEFT JOIN {parent} p "
            f"ON c.{column} = p.{column} WHERE c.{column} IS NOT NULL AND p.{column} IS NULL")

def run_tests(tests, extra_checks):
    results = []
    for model, columns in tests.items():
        for column, column_tests in columns.items():
            for test in column_tests:
                name = test if isinstance(test, str) else f"relationships→{test[1]}"
                failures = len(pd.read_sql(compile_test(model, column, test), con))
                results.append((model, column, name, failures))
    for name, sql in extra_checks.items():
        results.append(('custom', '', name, len(pd.read_sql(sql, con))))
    out = pd.DataFrame(results, columns=['model', 'column', 'test', 'failing_rows'])
    out['status'] = np.where(out.failing_rows == 0, '✅ PASS', '❌ FAIL')
    return out

extra_checks = {
    'fact rows == staging rows': '''
        SELECT 1 WHERE (SELECT COUNT(*) FROM fact_farm_yield) != (SELECT COUNT(*) FROM stg_farm_yield)''',
    'yield_tons > 0': "SELECT * FROM fact_farm_yield WHERE yield_tons <= 0",
}

results = run_tests(schema_tests, extra_checks)
print(f"{(results.failing_rows == 0).sum()} / {len(results)} tests passed")
results

### Do the tests catch real problems?

A test suite that always passes proves little. As a check on the tests themselves, I inject two defects into a copy of the fact table, an orphan foreign key and a negative yield, and confirm the right tests fail.

In [ ]:
con.execute("CREATE TABLE fact_farm_yield_backup AS SELECT * FROM fact_farm_yield")
con.execute('''INSERT INTO fact_farm_yield
               (farm_key, crop_key, region_key, soil_key, irrigation_key,
                fertilizer_used_kg, pesticide_used_kg, water_usage_m3, rainfall_mm, temperature_c, yield_tons)
               VALUES (99999, 1, 1, 1, 1, 100, 10, 500, 800, 25, -3.0)''')

broken = run_tests({'fact_farm_yield': schema_tests['fact_farm_yield']}, extra_checks)
display(broken[broken.failing_rows > 0])

# restore the clean fact table
con.executescript('''DROP TABLE fact_farm_yield;
                     ALTER TABLE fact_farm_yield_backup RENAME TO fact_farm_yield;''')

The orphan key trips `relationships→dim_farm`, the negative yield trips the range check, and the extra row trips the reconciliation check. The tests work.

## 7. Querying the mart (the Power BI step)

These are the kinds of queries the Power BI dashboard runs against the star schema.

In [ ]:
by_crop = pd.read_sql('''
    SELECT c.crop_type, COUNT(*) AS records, ROUND(AVG(f.yield_tons), 2) AS avg_yield_tons,
           ROUND(SUM(f.water_usage_m3) / SUM(f.yield_tons), 1) AS water_m3_per_ton
    FROM fact_farm_yield f JOIN dim_crop c USING (crop_key)
    GROUP BY c.crop_type ORDER BY avg_yield_tons DESC
''', con)

region_irrigation = pd.read_sql('''
    SELECT r.region_name, i.irrigation_type, AVG(f.yield_tons) AS avg_yield
    FROM fact_farm_yield f
    JOIN dim_region r USING (region_key)
    JOIN dim_irrigation i USING (irrigation_key)
    GROUP BY 1, 2
''', con).pivot(index='region_name', columns='irrigation_type', values='avg_yield')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(data=by_crop, x='crop_type', y='avg_yield_tons', ax=axes[0], color='#4C9A6A')
axes[0].set(title='Average yield by crop', xlabel='', ylabel='Tons')
sns.heatmap(region_irrigation, annot=True, fmt='.2f', cmap='YlGn', ax=axes[1])
axes[1].set(title='Average yield: region × irrigation', xlabel='', ylabel='')
plt.tight_layout()
plt.show()
by_crop

## 8. Sanity check: is there any signal in this data?

Before anyone tells a story from these charts (*"Drip irrigation in the North wins!"*), check whether the differences are bigger than chance. In the generator, `Yield` is drawn **independently** of every other column, so no real effects should exist.

In [ ]:
from scipy import stats

fact_wide = pd.read_sql('''
    SELECT c.crop_type, r.region_name, s.soil_type, i.irrigation_type, c.harvest_season,
           f.fertilizer_used_kg, f.pesticide_used_kg, f.water_usage_m3, f.rainfall_mm,
           f.temperature_c, f.yield_tons
    FROM fact_farm_yield f
    JOIN dim_crop c USING (crop_key) JOIN dim_region r USING (region_key)
    JOIN dim_soil s USING (soil_key) JOIN dim_irrigation i USING (irrigation_key)
''', con)

rows = []
for col in ['crop_type', 'region_name', 'soil_type', 'irrigation_type', 'harvest_season']:
    groups = [g.yield_tons.values for _, g in fact_wide.groupby(col)]
    rows.append((col, 'ANOVA', stats.f_oneway(*groups).pvalue))
for col in ['fertilizer_used_kg', 'pesticide_used_kg', 'water_usage_m3', 'rainfall_mm', 'temperature_c']:
    r, p = stats.pearsonr(fact_wide[col], fact_wide.yield_tons)
    rows.append((f"{col} (r = {r:+.3f})", 'Pearson', p))

signal = pd.DataFrame(rows, columns=['factor vs. yield', 'test', 'p_value'])
signal['significant at 0.05 (Bonferroni, 10 tests)'] = signal.p_value < 0.05 / len(signal)
signal.round(4)

As expected, none of the factors survive a multiple-comparison correction. The bars and heatmap cells above differ only through **sampling noise**. The pipeline works correctly, but a dashboard built on this data has no real story to tell. With real data, the same check separates actual effects from noise before they reach a report.

## 9. Summary

- ✅ Reproduced the **raw → staging → star schema** flow from the dbt project, with 5 dimensions and 1 fact table.
- ✅ Staging normalized dirty text and dropped a row with a null key without breaking the grain.
- ✅ Turned the `schema.yml` tests into SQL and added **row reconciliation** and **range** checks. Every test passes on clean data.
- ✅ **Tested the tests** by injecting defects, and the right checks failed.
- ✅ Ran a statistical **sanity check** before trusting the dashboard, which confirmed the synthetic data has no signal.

**Full project** (NiFi flow, Dockerized Airflow DAG, dbt project, Power BI): [GitHub — ZIADMAHMOUD0](https://github.com/ZIADMAHMOUD0)

If you found this useful, an upvote is appreciated 🙌